# 🌱 Recommendation Systems: Agricultural Input Recommendations for Farmers

**Author:** Jok Akech Atem Mabior | CMU Africa  
**Dataset:** Synthetic farmer-product interaction data  
**Goal:** Build collaborative filtering, matrix factorization, and hybrid recommendation systems for East African smallholder farmers.

## Background

Smallholder farmers in East Africa — who account for over 70% of food production — often lack access to tailored agronomic guidance. Key challenges include:

- **Information asymmetry:** Farmers may not know which seed varieties or fertilizers work best for their soil type and region
- **Input market fragmentation:** Hundreds of competing products (seeds, fertilizers, pesticides, tools) with varying quality and price points
- **Limited extension services:** Government agricultural extension officer ratios are often 1:2,000+ farmers

**Recommendation systems** can bridge this gap by:
- Learning from ratings and purchase behavior of similar farmers
- Personalizing input recommendations based on farm size, crop type, and region
- Helping agri-input retailers (like Twiga Foods, Apollo Agriculture) suggest optimal products

This notebook implements and compares **User-Based Collaborative Filtering**, **Matrix Factorization (SVD/NMF)**, **Content-Based Filtering**, and a **Hybrid system**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD, NMF
from sklearn.preprocessing import normalize, LabelEncoder
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8')
print("Libraries loaded!")

## 1. Data Generation

In [ ]:
np.random.seed(42)
N_FARMERS = 400
N_PRODUCTS = 40

product_categories = {
    'Maize Seeds': ['DK8031', 'H614D', 'WH403', 'SC403', 'SEEDCO'],
    'Bean Seeds': ['Rosecoco', 'KK8', 'GLP2', 'Lyamungu85', 'Jesca'],
    'Fertilizer': ['CAN', 'DAP', 'NPK17-17-17', 'Urea', 'CuSO4'],
    'Pesticide': ['Bulldock', 'Karate', 'Confidor', 'Actara', 'Dimethoate'],
    'Herbicide': ['Roundup', 'Weedmaster', 'Touchdown', 'Lasso', 'Trophy'],
    'Irrigation': ['Drip Kit 0.1ha', 'Sprinkler Set', 'Manual Pump', 'Treadle Pump', 'Rain Gun'],
    'Tools': ['Jembe Set', 'Panga', 'Wheelbarrow', 'Knapsack Sprayer', 'Seeder'],
    'Storage': ['Grain Bag 100kg', 'PICS Bag', 'Metal Silo 500kg', 'Hermetic Bucket', 'Moisture Meter']
}

all_products = [p for cat_products in product_categories.values() for p in cat_products]
product_category_map = {p: cat for cat, products in product_categories.items() for p in products}

countries = ['Kenya', 'Uganda', 'Tanzania', 'Rwanda', 'Ethiopia']
farmer_country = np.random.choice(countries, N_FARMERS)
farmer_farm_size = np.random.exponential(2.5, N_FARMERS).clip(0.5, 20)
farmer_crop_focus = np.random.choice(['Maize', 'Beans', 'Mixed', 'Vegetables', 'Coffee'], N_FARMERS, p=[0.35, 0.2, 0.25, 0.1, 0.1])

farmers = pd.DataFrame({
    'farmer_id': [f'F{i:04d}' for i in range(N_FARMERS)],
    'country': farmer_country,
    'farm_size_ha': farmer_farm_size.round(2),
    'crop_focus': farmer_crop_focus
})

rating_matrix = np.zeros((N_FARMERS, N_PRODUCTS))

for farmer_idx in range(N_FARMERS):
    n_ratings = np.random.randint(4, 13)
    crop = farmer_crop_focus[farmer_idx]
    probs = np.ones(N_PRODUCTS) / N_PRODUCTS
    for p_idx, p in enumerate(all_products):
        cat = product_category_map[p]
        if crop == 'Maize' and 'Maize' in cat: probs[p_idx] *= 5
        elif crop == 'Beans' and 'Bean' in cat: probs[p_idx] *= 5
        if cat in ['Fertilizer', 'Tools', 'Storage']: probs[p_idx] *= 2
    probs /= probs.sum()
    rated_products = np.random.choice(N_PRODUCTS, n_ratings, replace=False, p=probs)
    for p_idx in rated_products:
        base_rating = np.random.choice([1, 2, 3, 4, 5], p=[0.05, 0.10, 0.20, 0.35, 0.30])
        rating_matrix[farmer_idx, p_idx] = base_rating

products = pd.DataFrame({
    'product_id': [f'P{i:03d}' for i in range(N_PRODUCTS)],
    'product_name': all_products,
    'category': [product_category_map[p] for p in all_products]
})

print(f"Farmers: {N_FARMERS}, Products: {N_PRODUCTS}")
print(f"Rating matrix: {rating_matrix.shape}")
print(f"Sparsity: {(rating_matrix == 0).mean()*100:.1f}% zeros")
print(f"Ratings distribution:")
print(pd.Series(rating_matrix[rating_matrix > 0].flatten().astype(int)).value_counts().sort_index())

## 2. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

all_ratings = rating_matrix[rating_matrix > 0].flatten()
axes[0].hist(all_ratings, bins=5, edgecolor='white', color='#27ae60', alpha=0.8, rwidth=0.8)
axes[0].set_xticks([1,2,3,4,5])
axes[0].set_xlabel('Rating (1-5)')
axes[0].set_ylabel('Count')
axes[0].set_title('Rating Distribution', fontweight='bold')

ratings_per_product = (rating_matrix > 0).sum(axis=0)
product_rating_df = pd.DataFrame({'product': all_products, 'n_ratings': ratings_per_product,
                                   'category': [product_category_map[p] for p in all_products]})
product_rating_df = product_rating_df.sort_values('n_ratings', ascending=False)
axes[1].barh(product_rating_df['product'].head(15)[::-1],
              product_rating_df['n_ratings'].head(15)[::-1],
              color='steelblue', alpha=0.8)
axes[1].set_title('Top 15 Most Rated Products', fontweight='bold')
axes[1].set_xlabel('Number of Ratings')

cat_ratings = product_rating_df.groupby('category')['n_ratings'].sum().sort_values(ascending=False)
axes[2].barh(cat_ratings.index, cat_ratings.values, color=sns.color_palette('husl', len(cat_ratings)))
axes[2].set_title('Total Ratings by Product Category', fontweight='bold')
axes[2].set_xlabel('Total Ratings')

plt.tight_layout()
plt.show()
print(f"Total ratings recorded: {(rating_matrix > 0).sum()}")
print(f"Average ratings per farmer: {(rating_matrix > 0).sum(axis=1).mean():.1f}")

In [ ]:
sample_farmers = np.random.choice(N_FARMERS, 30, replace=False)

plt.figure(figsize=(18, 8))
sns.heatmap(rating_matrix[sample_farmers][:, :N_PRODUCTS],
            cmap='YlOrRd', cbar_kws={'label': 'Rating (0=not rated)'},
            xticklabels=all_products, yticklabels=[f'F{i}' for i in sample_farmers])
plt.title('Farmer-Product Rating Matrix (30 farmers sample)', fontweight='bold')
plt.xlabel('Product')
plt.ylabel('Farmer')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.show()

## 3. Collaborative Filtering — User-Based

User-based collaborative filtering finds farmers with similar rating patterns and uses their preferences to predict unrated products.

In [ ]:
user_similarity = cosine_similarity(rating_matrix)
np.fill_diagonal(user_similarity, 0)

print(f"User similarity matrix: {user_similarity.shape}")
print(f"Mean user-user similarity: {user_similarity[user_similarity > 0].mean():.4f}")

def user_based_predict(user_idx, product_idx, k=10):
    similarities = user_similarity[user_idx]
    rated_mask = rating_matrix[:, product_idx] > 0
    if rated_mask.sum() == 0:
        return rating_matrix[rating_matrix > 0].mean()
    similar_users = similarities * rated_mask
    top_k_idx = similar_users.argsort()[::-1][:k]
    top_k_sim = similar_users[top_k_idx]
    top_k_ratings = rating_matrix[top_k_idx, product_idx]
    valid = top_k_sim > 0
    if valid.sum() == 0:
        return rating_matrix[rating_matrix > 0].mean()
    return np.average(top_k_ratings[valid], weights=top_k_sim[valid])

test_farmer = 5
print(f"\nFarmer {farmers.iloc[test_farmer]['farmer_id']} profile:")
print(f"  Country: {farmers.iloc[test_farmer]['country']}")
print(f"  Crop focus: {farmers.iloc[test_farmer]['crop_focus']}")
print(f"  Farm size: {farmers.iloc[test_farmer]['farm_size_ha']} ha")
already_rated = np.where(rating_matrix[test_farmer] > 0)[0]
print(f"  Already rated: {[all_products[i] for i in already_rated]}")

In [ ]:
def get_recommendations(farmer_idx, n=5, k=10):
    unrated = np.where(rating_matrix[farmer_idx] == 0)[0]
    predictions = []
    for p_idx in unrated:
        pred = user_based_predict(farmer_idx, p_idx, k)
        predictions.append((p_idx, pred, all_products[p_idx], product_category_map[all_products[p_idx]]))
    predictions.sort(key=lambda x: x[1], reverse=True)
    return predictions[:n]

recs = get_recommendations(test_farmer, n=8)
print(f"Top 8 Recommendations for Farmer {farmers.iloc[test_farmer]['farmer_id']}:")
rec_df = pd.DataFrame(recs, columns=['product_idx', 'predicted_rating', 'product', 'category'])
print(rec_df[['product', 'category', 'predicted_rating']].to_string(index=False))

## 4. Matrix Factorization (SVD)

Matrix factorization decomposes the rating matrix into latent farmer and product factors, capturing hidden preferences and product attributes.

In [ ]:
rating_filled = rating_matrix.copy()
mean_rating = rating_matrix[rating_matrix > 0].mean()
rating_filled[rating_filled == 0] = mean_rating

n_components = 15
svd = TruncatedSVD(n_components=n_components, random_state=42)
U = svd.fit_transform(rating_filled)
Vt = svd.components_

rating_pred = U @ Vt
print(f"SVD explained variance: {svd.explained_variance_ratio_.sum()*100:.1f}%")
print(f"Reconstruction matrix: {rating_pred.shape}")

plt.figure(figsize=(10, 4))
plt.bar(range(1, n_components+1), svd.explained_variance_ratio_*100, color='steelblue', alpha=0.8)
plt.xlabel('SVD Component')
plt.ylabel('Explained Variance (%)')
plt.title('SVD — Variance Captured per Latent Factor', fontweight='bold')
plt.xticks(range(1, n_components+1))
plt.tight_layout()
plt.show()

In [ ]:
known_ratings = []
for i in range(N_FARMERS):
    for j in range(N_PRODUCTS):
        if rating_matrix[i, j] > 0:
            known_ratings.append((i, j, rating_matrix[i, j]))

np.random.shuffle(known_ratings)
split = int(0.8 * len(known_ratings))
test_ratings = known_ratings[split:]

svd_errors = []
ucf_errors = []
for farmer_idx, prod_idx, actual in test_ratings[:200]:
    svd_pred = np.clip(rating_pred[farmer_idx, prod_idx], 1, 5)
    ucf_pred = np.clip(user_based_predict(farmer_idx, prod_idx), 1, 5)
    svd_errors.append((svd_pred - actual)**2)
    ucf_errors.append((ucf_pred - actual)**2)

baseline_rmse = np.sqrt(np.mean([(mean_rating - r[2])**2 for r in test_ratings[:200]]))
print(f"Test set evaluation (200 samples):")
print(f"  SVD RMSE:      {np.sqrt(np.mean(svd_errors)):.4f}")
print(f"  User-CF RMSE:  {np.sqrt(np.mean(ucf_errors)):.4f}")
print(f"  Baseline RMSE: {baseline_rmse:.4f}")

methods = ['SVD', 'User-CF', 'Baseline (Mean)']
rmses = [np.sqrt(np.mean(svd_errors)), np.sqrt(np.mean(ucf_errors)), baseline_rmse]
plt.figure(figsize=(8, 4))
plt.bar(methods, rmses, color=['#3498db', '#2ecc71', '#e74c3c'], alpha=0.85, edgecolor='white')
plt.ylabel('RMSE')
plt.title('Recommender System RMSE Comparison', fontweight='bold')
for i, v in enumerate(rmses):
    plt.text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Content-Based Filtering

Content-based filtering uses farmer profile features (farm size, crop focus, country) and product category features to compute similarity scores without requiring historical ratings.

In [ ]:
crop_dummies = pd.get_dummies(farmers['crop_focus'], prefix='crop')
country_dummies = pd.get_dummies(farmers['country'], prefix='country')
farmer_features = pd.concat([farmers[['farm_size_ha']], crop_dummies, country_dummies], axis=1).values

cat_dummies = pd.get_dummies(products['category'], prefix='cat')
product_features = cat_dummies.values

farmer_norm = normalize(farmer_features.astype(float))
product_norm = normalize(product_features.astype(float))

content_scores = farmer_norm @ product_norm.T
print(f"Content-based score matrix: {content_scores.shape}")

unrated_mask = rating_matrix[test_farmer] == 0
cb_scores_farmer = content_scores[test_farmer] * unrated_mask
top_cb = cb_scores_farmer.argsort()[::-1][:5]
print(f"\nContent-Based Recommendations for Farmer {farmers.iloc[test_farmer]['farmer_id']}:")
for idx in top_cb:
    print(f"  {all_products[idx]:25s} ({product_category_map[all_products[idx]]}) — Score: {cb_scores_farmer[idx]:.4f}")

In [ ]:
alpha = 0.6
beta  = 0.4

svd_norm = (rating_pred - rating_pred.min()) / (rating_pred.max() - rating_pred.min())
cb_norm  = (content_scores - content_scores.min()) / (content_scores.max() - content_scores.min())

hybrid_scores = alpha * svd_norm + beta * cb_norm

print("Hybrid Recommendations for first 5 farmers:")
for farmer_idx in range(5):
    unrated = rating_matrix[farmer_idx] == 0
    scores = hybrid_scores[farmer_idx] * unrated
    top_3 = scores.argsort()[::-1][:3]
    print(f"\n  {farmers.iloc[farmer_idx]['farmer_id']} ({farmers.iloc[farmer_idx]['crop_focus']}, {farmers.iloc[farmer_idx]['country']}):")
    for p in top_3:
        print(f"    - {all_products[p]} ({product_category_map[all_products[p]]}) — Hybrid score: {scores[p]:.3f}")

## 6. NMF-Based Factorization

Non-negative Matrix Factorization (NMF) produces parts-based representations — useful here as ratings are non-negative and NMF components can be interpreted as latent 'farming profiles'.

In [ ]:
nmf = NMF(n_components=10, random_state=42, max_iter=500)
W = nmf.fit_transform(rating_filled)
H = nmf.components_
rating_pred_nmf = W @ H

nmf_errors = []
for farmer_idx, prod_idx, actual in test_ratings[:200]:
    nmf_pred = np.clip(rating_pred_nmf[farmer_idx, prod_idx], 1, 5)
    nmf_errors.append((nmf_pred - actual)**2)

print(f"NMF RMSE: {np.sqrt(np.mean(nmf_errors)):.4f}")
print(f"NMF reconstruction error: {nmf.reconstruction_err_:.4f}")

# Visualize NMF latent factors (farmer profiles)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(W[:50, :], aspect='auto', cmap='YlGn')
axes[0].set_title('NMF Farmer Latent Factors (50 farmers)', fontweight='bold')
axes[0].set_xlabel('Latent Factor')
axes[0].set_ylabel('Farmer')
axes[0].set_xticks(range(10))

axes[1].imshow(H, aspect='auto', cmap='OrRd')
axes[1].set_title('NMF Product Latent Factors', fontweight='bold')
axes[1].set_xlabel('Product')
axes[1].set_ylabel('Latent Factor')
axes[1].set_yticks(range(10))
plt.tight_layout()
plt.show()

## 7. Full System Comparison & Precision@K

In [ ]:
# Compare all approaches on RMSE
all_rmse = {
    'User-CF': np.sqrt(np.mean(ucf_errors)),
    'SVD': np.sqrt(np.mean(svd_errors)),
    'NMF': np.sqrt(np.mean(nmf_errors)),
    'Baseline': baseline_rmse
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette = sns.color_palette('viridis', len(all_rmse))
bars = axes[0].bar(all_rmse.keys(), all_rmse.values(), color=palette, edgecolor='white', alpha=0.85)
axes[0].set_title('RMSE Comparison Across Methods', fontweight='bold')
axes[0].set_ylabel('RMSE (lower is better)')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                 f'{bar.get_height():.4f}', ha='center', fontsize=10, fontweight='bold')

# Farmer similarity distribution
sim_values = user_similarity[user_similarity > 0].flatten()
axes[1].hist(sim_values, bins=50, color='steelblue', edgecolor='none', alpha=0.8)
axes[1].set_title('User-User Cosine Similarity Distribution', fontweight='bold')
axes[1].set_xlabel('Cosine Similarity')
axes[1].set_ylabel('Count')
axes[1].axvline(sim_values.mean(), color='red', ls='--', lw=2, label=f'Mean={sim_values.mean():.3f}')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nFinal RMSE Summary:")
for method, rmse in sorted(all_rmse.items(), key=lambda x: x[1]):
    print(f"  {method:15s}: RMSE = {rmse:.4f}")

## 8. Conclusions

### Key Findings

1. **Collaborative Filtering** performs well when there are enough similar farmers, but struggles with the **cold-start problem** for new farmers with no rating history.

2. **SVD Matrix Factorization** provides the best RMSE by capturing latent farmer-product preference patterns that transcend individual ratings, generalizing better to unseen farmer-product pairs.

3. **Content-Based Filtering** is valuable precisely in cold-start scenarios — it can recommend products to new farmers based solely on their profile (crop type, farm size, region).

4. **Hybrid Systems** (SVD + Content-Based) offer the best of both worlds: personalized recommendations with cold-start robustness.

### Business Value for East African Agriculture
- **Apollo Agriculture (Kenya):** Personalize loan and input packages per farmer
- **Twiga Foods:** Recommend complementary agrochemical products to increase basket size
- **USSD-based agricultural platforms:** Serve recommendations to feature phone users

### Limitations
- Synthetic ratings do not capture real farmer preferences or seasonal variation
- No temporal dynamics (preferences change across seasons)
- Ratings sparsity (only ~15% filled) limits collaborative filtering accuracy

### Next Steps
- **Deep learning rec systems:** Neural Collaborative Filtering (NCF), two-tower models
- **Context-aware recommendations:** Season, rainfall forecasts, soil type
- **A/B testing:** Deploy hybrid system and measure adoption rate of recommended products
- **Implicit feedback:** Use purchase history rather than explicit ratings